In [20]:
import pandas as pd
import os
import pickle

import sys
sys.path.insert(0, "../../..")

from utils.data_processing import add_boundary_column

In [21]:
df = pd.read_csv("../dataset_construction/datasets/processed_aemri75.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])
df = add_boundary_column(df, lang="ae")

In [22]:
distance_df = pd.read_csv("../distinctiveness_calculation/distances_aemri75.csv")
surprisal_df = pd.read_csv("../surprisal_calculation/surprisal_aemri75.csv")

In [23]:
for col_name in distance_df.columns:
    df[col_name] = distance_df[col_name]
    
for col_name in surprisal_df.columns:
    df[col_name] = surprisal_df[col_name]

In [24]:
with open("../../entropy_estimation/entropies/aemri75_art_given_ac.pickle", "rb") as f:
    H_art_given_ac_mdn = pickle.load(f)
with open("../../entropy_estimation/entropies/aemri75_ac_given_art.pickle", "rb") as f:
    H_ac_given_art_mdn = pickle.load(f)

In [25]:
df["phone1_H_art_given_ac"] = df.apply(lambda x: H_art_given_ac_mdn.get(x["phone1"], None), axis=1)
df["phone2_H_art_given_ac"] = df.apply(lambda x: H_art_given_ac_mdn.get(x["phone2"], None), axis=1)
df["phone1_H_ac_given_art"] = df.apply(lambda x: H_ac_given_art_mdn.get(x["phone1"], None), axis=1)
df["phone2_H_ac_given_art"] = df.apply(lambda x: H_ac_given_art_mdn.get(x["phone2"], None), axis=1)

In [26]:
dist_cols = [
    "ac_latent_dist",
    "ac_raw_dist",
    "art_latent_dist",
    "art_raw_dist"
]

df_long = df.melt(
    id_vars=[col for col in df.columns if col not in dist_cols],
    value_vars=dist_cols,
    var_name="dist_type",
    value_name="dist"
)

df_long[["modality", "data_space"]] = (
    df_long["dist_type"]
    .str.extract(r"^(ac|art)_(latent|raw)_dist$")
)

df_long = df_long.drop(columns="dist_type")
df_long.to_csv("results_aemri75.csv", index=False)

In [27]:
df_aemri = pd.read_csv("results_aemri.csv")
df_aemri75 = pd.read_csv("results_aemri75.csv")

In [28]:
df_aemri["dataset"] = "aemri"
df_aemri75["dataset"] = "aemri75"

In [29]:
pd.concat([df_aemri, df_aemri75], ignore_index=True).to_csv("results_aemri_combined.csv", index=False)